# A Regression Model

In brief, such models say that the underlying relation between the two variables is perfectly linear; this straight line is the *signal* that we would like to identify. However, we are not able to see the line clearly. What we see are points that are scattered around the line. In each of the points, the signal has been contaminated by *random noise*. Our inferential goal, therefore, is to separate the signal from the noise.

In greater detail, the regression model specifies that the points in the scatter plot are generated at random as follows.

- The relation between $x$ and $y$ is perfectly linear. We cannot see this "true line" but it exists.
- The scatter plot is created by taking points on the line and pushing them off the line vertically, either above or below, as follows:
    - For each $x$, find the corresponding point on the true line (that's the signal), and then generate the noise or error.
    - The errors are drawn at random with replacement from a population of errors that has a normal distribution with mean 0.
    - Create a point whose horizontal coordinate is $x$ and whose vertical coordinate is "the height of the true line at $x$, plus the error".
- Finally, erase the true line from the scatter, and display just the points created.

In [1]:
import polars as pl
import numpy as np
from datascience import make_array
import hvplot.polars
import holoviews as hv
path_data = '../../../assets/data/'

In [2]:
def standard_units(any_numbers):
    "Convert any Series of numbers to standard units."
    return (any_numbers - any_numbers.mean())/any_numbers.std(ddof=0)  

def correlation(t, x, y):
    return (standard_units(t.get_column(x))*standard_units(t.get_column(y))).mean()

def slope(table, x, y):
    r = correlation(table, x, y)
    return r * table.get_column(y).std(ddof=0)/table.get_column(x).std(ddof=0)

def intercept(table, x, y):
    a = slope(table, x, y)
    return table.get_column(y).mean() - a * table.get_column(x).mean()

def fit(table, x, y):
    a = slope(table, x, y)
    b = intercept(table, x, y)
    return a * table.get_column(x) + b

def scatter_fit(table, x, y):
    """POLARS_CHANGE: hvplot.scatter with the regression line overlaid in yellow with a thin black outline. The chart is returned so a cell can show it or draw on it."""
    points = table.hvplot.scatter(x=x, y=y, color='#002fa7')
    fitted = pl.DataFrame([table.get_column(x), fit(table, x, y).alias('Fitted value')]).sort(x)
    line = (
        fitted.hvplot.line(x=x, y='Fitted value', color='black', line_width=3)  # dark outline for contrast
        * fitted.hvplot.line(x=x, y='Fitted value', color='#FFFF00', line_width=2)
    )
    return (points * line).opts(xlabel=x, ylabel=y)

Based on this scatter plot, how should we estimate the true line? The best line that we can put through a scatter plot is the regression line. So the regression line is a natural estimate of the true line. 

The simulation below shows how close the regression line is to the true line. The first panel shows how the scatter plot is generated from the true line. The second shows the scatter plot that we see. The third shows the regression line through the plot. The fourth shows both the regression line and the true line.

To run the simulation, call the function `draw_and_compare` with three arguments: the slope of the true line, the intercept of the true line, and the sample size.

Run the simulation a few times, with different values for the slope and intercept of the true line, and varying sample sizes. Because all the points are generated according to the model, you will see that the regression line is a good estimate of the true line if the sample size is moderately large.

In [3]:
def draw_and_compare(true_slope, true_int, sample_size):
    x = np.random.normal(50, 5, sample_size)
    xlims = make_array(np.min(x), np.max(x))
    eps = np.random.normal(0, 6, sample_size)
    y = (true_slope*x + true_int) + eps
    tyche = pl.DataFrame([
        pl.Series('x', x),
        pl.Series('y', y),
    ])

    true_line = hv.Curve((xlims, true_slope*xlims + true_int)).opts(color='#1e6400', line_width=2)
    graph_1 = (tyche.hvplot.scatter(x='x', y='y', color='#002fa7') * true_line).opts(title='True Line, and Points Created')

    graph_2 = tyche.hvplot.scatter(x='x', y='y', color='#002fa7').opts(title='What We Get to See')

    graph_3 = scatter_fit(tyche, 'x', 'y').opts(xlabel='', ylabel='', title='Regression Line: Estimate of True Line')

    xlims = make_array(tyche.get_column('x').min(), tyche.get_column('x').max())
    true_line = hv.Curve((xlims, true_slope*xlims + true_int)).opts(color='#1e6400', line_width=2)
    graph_4 = (scatter_fit(tyche, 'x', 'y') * true_line).opts(ylabel='', title='Regression Line and True Line')

    # POLARS_CHANGE: the four matplotlib subplots are a HoloViews layout with one graph per row
    return (graph_1 + graph_2 + graph_3 + graph_4).cols(1)

In [4]:
# The true line,
# the points created,
# and our estimate of the true line.
# Arguments: true slope, true intercept, number of points

draw_and_compare(4, -5, 10)

Four graphs are shown. The first graph is titled 'True Line, and Points Created' and has a green line with a positive slope and data points in light blue. The second plot is titled 'What We Get to See' with the blue data points shown. The third plot is titled 'Regression Line: Estimate of True Line' and the light blue data points remain along with a yellow line that is much like the green line of the first graph, but slightly different. The fourth graph is titled 'Regression Line and True Line' and shows the data points, the yellow line, and the green line. The lines are very similar, but slightly different.

In reality, of course, we will never see the true line. What the simulation shows that if the regression model looks plausible, and if we have a large sample, then the regression line is a good approximation to the true line.